In [1]:
import pandas as pd
import numpy as np
import requests
import re
import spacy
from spacy import displacy
nlp = spacy.load('en_core_web_sm')

#### Find average stock price from text data

In [2]:
url = 'https://cdn.upgrad.com/uploads/production/83a0597f-1dd2-4653-8194-cbef1870e5fb/data.txt'
response = requests.get(url)

In [3]:
data = response.text
data

"The stock price of a certain company was $100 a year ago, once this company came into boom phase then it's stock price rise to $200. On the other hand, when there was economic depression the stock price was $50. When the company had started the price of its stock was $10, which you can consider as the base price. The stock price of a certain company was $100 a year ago, once this company came into boom phase then it's stock price rise to $200. On the other hand, when there was economic depression the stock price was $50. When the company had started the price of its stock was $10, which you can consider as the base price. The stock price of a certain company was $100 a year ago, once this company came into boom phase then it's stock price rise to $200. On the other hand, when there was economic depression the stock price was $50. When the company had started the price of its stock was $10, which you can consider as the base price. The stock price of a certain company was $100 a year a

In [121]:
# List of NER entities in spacy
nlp.get_pipe('ner').labels

('CARDINAL',
 'DATE',
 'EVENT',
 'FAC',
 'GPE',
 'LANGUAGE',
 'LAW',
 'LOC',
 'MONEY',
 'NORP',
 'ORDINAL',
 'ORG',
 'PERCENT',
 'PERSON',
 'PRODUCT',
 'QUANTITY',
 'TIME',
 'WORK_OF_ART')

In [122]:
money_values = np.array([float(tok.text) for tok in nlp(data) if tok.ent_type_ == 'MONEY'])
money_values.mean()

np.float64(93.25806451612904)

Date names from text data

In [123]:
date_names = list(set([' '.join([tok.text] + [str(w) for w in list(tok.ancestors)]) for tok in nlp(data) 
              if tok.ent_type_ == 'DATE'
              ]))
date_names

["year ago was 's", "a year ago was 's", "ago was 's"]

In [124]:
# Visualizing date pos dependencies
for doc in date_names:
    displacy.render(nlp(doc), style='dep')

Custom CRF functions

In [125]:
# Training corpus
sents = [
    'Bella is not giving money',
    'Tony is teaching NLP to Chris',
    'Joe should focus to his job',
    'Peter is trained by Adele',
    'Temperature is maintained by AC',
    'He is playing the Cricket',
    'Guitar has been played by him'
]

labels = [
    ['P', '', '', 'A', 'E'],
    ['P', '', 'A', 'E', '', 'P'],
    ['P', '', 'A', '', '', 'E'],
    ['P', '', 'A', '', 'P'],
    ['E', '', 'A', '', 'E'],
    ['', '', 'A', '', 'E'],
    ['E', '', '', 'A', '', 'V']
]

In [126]:
# Feature functions
def GetFeatureSent(sent, label):
    
    caps = lambda x: 1 if re.match('Xx+', x) else 0
    f1 = [caps(tok.shape_) for tok in nlp(sent)]
    
    nsub_EP = lambda x: 1 if len([tokc.dep_ for tokc in list(x.children) 
                            if tokc.dep_ in ['nsubj', 'nsubjpass']
                            and label[tokc.i] in ['E', 'P']])>0 else 0
    f2 = [nsub_EP(tok) for tok in nlp(sent)]

    obj_EP = lambda x: 1 if len([tokc.dep_ for tokc in list(x.children)
                             if tokc.dep_ == 'dobj'
                             and label[tokc.i] in ['E', 'P']])>0 else 0
    f3 = [obj_EP(tok) for tok in nlp(sent)]

    return np.array([f1, f2, f3])

In [ ]:
# Create dataframe to visualize
sent = 'Harry is not gardening as it is raining'
label = ['P', '', '', 'A', '', '', '', 'A']

pd.DataFrame(GetFeatureSent(sent, label), columns=sent.split(), index=['F1', 'F2', 'F3'])

,Harry,is,not,gardening,as,it,is,raining
F1,1,0,0,0,0,0,0,0
F2,0,0,0,1,0,0,0,0
F3,0,0,0,0,0,0,0,0


IOB labels

In [135]:
sent = 'What is the price of American Airlines flight from New York to Los Angeles'

pd.DataFrame([{'Text':tok.text, 'IOB':f'{str(tok.ent_iob_)}-{str(tok.ent_type_)}'
               if len(tok.ent_type_)>0 else f'{str(tok.ent_iob_)}'} 
              for tok in nlp(sent)])

,Text,IOB
0,What,O
1,is,O
2,the,O
3,price,O
4,of,O
5,American,B-ORG
6,Airlines,I-ORG
7,flight,O
8,from,O
9,New,B-GPE


In [6]:
sents = data.split('.')

In [9]:
[(tok.text, tok.pos) for tok in nlp(sents[10])]

[(' ', 103),
 ('On', 85),
 ('the', 90),
 ('other', 84),
 ('hand', 92),
 (',', 97),
 ('when', 98),
 ('there', 95),
 ('was', 100),
 ('economic', 84),
 ('depression', 92),
 ('the', 90),
 ('stock', 92),
 ('price', 92),
 ('was', 87),
 ('$', 99),
 ('50', 93)]

In [ ]:
# List of POS tags
nlp.get_pipe('tagger').labels

('$',
 "''",
 ',',
 '-LRB-',
 '-RRB-',
 '.',
 ':',
 'ADD',
 'AFX',
 'CC',
 'CD',
 'DT',
 'EX',
 'FW',
 'HYPH',
 'IN',
 'JJ',
 'JJR',
 'JJS',
 'LS',
 'MD',
 'NFP',
 'NN',
 'NNP',
 'NNPS',
 'NNS',
 'PDT',
 'POS',
 'PRP',
 'PRP$',
 'RB',
 'RBR',
 'RBS',
 'RP',
 'SYM',
 'TO',
 'UH',
 'VB',
 'VBD',
 'VBG',
 'VBN',
 'VBP',
 'VBZ',
 'WDT',
 'WP',
 'WP$',
 'WRB',
 'XX',
 '_SP',
 '``')